# SCOUT on a 2-D toy

This notebook is self-contained. Every function it uses is defined below. It does not clone the repository and it does not import the `scout` package.

The real failure regions are two diamonds in \([-3, 3]^2\). Inside a diamond the target score is high. The proxy is the same field with a small shift, and the second diamond is almost missing from the proxy. SCOUT uses that proxy only as a control variate: mutual information proposes where the current labels are uninformative, and the proxy re-ranks that shortlist.


## 1. Install


In [ ]:
import importlib.util, subprocess, sys
needed = ("numpy", "sklearn", "matplotlib")
missing = [name for name in needed if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "scikit-learn", "matplotlib"])
print("ready")


## 2. Domain

A design is a point \(x = (x_1, x_2)\). `real_mean` is the target. `sim_mean` is the cheap proxy. Real observations add a little noise. The proxy is free everywhere.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
})

C1 = np.array([1.35, 1.35])
C2 = np.array([-1.35, 1.35])
R1, R2 = 0.65, 0.55

def diamond_score(x, center, radius):
    return radius - (np.abs(x[..., 0] - center[0]) + np.abs(x[..., 1] - center[1]))

def real_mean(x):
    base = np.maximum(diamond_score(x, C1, R1), diamond_score(x, C2, R2))
    return 1.15 * np.tanh(2.4 * base) + 0.10 * np.sin(1.1 * x[..., 0]) - 0.08 * np.cos(0.9 * x[..., 1])

def sim_mean(x):
    xs = np.array(x, dtype=float, copy=True)
    xs[..., 0] = xs[..., 0] + 0.18
    xs[..., 1] = xs[..., 1] - 0.12
    base = np.maximum(diamond_score(xs, C1, R1 * 1.05), 0.1 * diamond_score(xs, C2, R2 * 0.95))
    return 1.15 * np.tanh(2.4 * base) + 0.10 * np.sin(1.1 * x[..., 0]) - 0.08 * np.cos(0.9 * x[..., 1])

def sample_real(x, seed):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    return real_mean(x) + np.sqrt(0.015) * rng.standard_normal(len(x))

def make_grid(n=48):
    g = np.linspace(-3, 3, n)
    xx, yy = np.meshgrid(g, g)
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1)
    return grid

def diamond_outline(center, radius):
    cx, cy = float(center[0]), float(center[1])
    return np.array([[cx + radius, cy], [cx, cy + radius], [cx - radius, cy], [cx, cy - radius], [cx + radius, cy]])

def draw_diamonds(ax):
    for center, radius in ((C1, R1), (C2, R2)):
        v = diamond_outline(center, radius)
        ax.plot(v[:, 0], v[:, 1], color="k", lw=1.0)

def heat(ax, grid, values, title):
    n = int(np.sqrt(len(grid)))
    im = ax.imshow(values.reshape(n, n), origin="lower", extent=[-3, 3, -3, 3], cmap="magma", aspect="equal")
    draw_diamonds(ax)
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    return im


## 3. Step 1 — real field and proxy

The outlines are the real diamonds. The right-hand diamond survives in the proxy. The left-hand diamond is much weaker there, which is the sim-to-real gap this example is built around.


In [ ]:
grid = make_grid(56)
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), constrained_layout=True)
im0 = heat(axes[0], grid, real_mean(grid), "Real target")
im1 = heat(axes[1], grid, sim_mean(grid), "Proxy (shifted, second diamond weakened)")
fig.colorbar(im0, ax=axes[0], fraction=0.046)
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.show()


## 4. Step 2 — a few real labels

White points are the only places where the real score has been observed. The proxy can still be evaluated at every other design for free.


In [ ]:
SEED, N_INIT = 7, 16
rng = np.random.default_rng(SEED)
x_init = rng.uniform(-3, 3, size=(N_INIT, 2))
y_init = sample_real(x_init, seed=SEED + 1)

fig, ax = plt.subplots(figsize=(5.2, 4.6), constrained_layout=True)
im = heat(ax, grid, real_mean(grid), "Initial real labels")
ax.scatter(x_init[:, 0], x_init[:, 1], c="white", edgecolors="k", s=36, zorder=3)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.show()


## 5. Step 3 — mutual-information shortlist

Cluster the labeled designs. A candidate far from those clusters is likely to reveal a new region, so its mutual information is high. The shortlist is the top of that score among random designs inside the box, not the corners of a plotting grid.


In [ ]:
def _normalize(p, eps=1e-12):
    p = np.clip(np.asarray(p, dtype=float), 0.0, None)
    s = p.sum()
    return p / s if s >= eps else np.ones_like(p) / len(p)

def _kl(p, q, eps=1e-12):
    p = np.clip(_normalize(p), eps, 1.0)
    q = np.clip(_normalize(q), eps, 1.0)
    return float(np.sum(p * np.log(p / q)))

def _sigmoid(z):
    z = np.clip(z, -60, 60)
    return 1.0 / (1.0 + np.exp(-z))

def mutual_information(x_support, x_query, n_clusters=6, pi_new=0.18, seed=0):
    n_eff = min(n_clusters, len(x_support))
    km = KMeans(n_clusters=n_eff, random_state=seed, n_init="auto")
    labels = km.fit_predict(x_support)
    info = []
    for k, center in enumerate(km.cluster_centers_):
        pts = x_support[labels == k]
        dists = np.linalg.norm(pts - center, axis=1) if len(pts) else np.array([1e-6])
        info.append(dict(center=center, radius=max(float(np.quantile(dists, 0.9)), 1e-6),
                         scale=max(float(np.std(dists) + 1e-6), 1e-6), size=max(len(pts), 1)))
    sizes = np.array([c["size"] for c in info], dtype=float)
    pZ = _normalize(np.concatenate([(1.0 - pi_new) * sizes / sizes.sum(), [pi_new]]))

    def dist_to_ball(x, c):
        return max(0.0, float(np.linalg.norm(x - c["center"])) - c["radius"])

    def likelihood_r1(x):
        q = []
        for c in info:
            d = dist_to_ball(x, c)
            tau = 1.3 * c["scale"]
            q.append(0.04 * (1.0 - np.exp(-0.5 * (d / tau) ** 2)))
        dmin = min(dist_to_ball(x, c) for c in info)
        q.append(float(_sigmoid((dmin - 0.75) / 0.25)))
        return np.asarray(q, dtype=float)

    mi = np.zeros(len(x_query))
    for i, x in enumerate(x_query):
        q1 = likelihood_r1(x)
        p_r1 = float(np.sum(pZ * q1))
        p_r0 = 1.0 - p_r1
        post1 = _normalize(q1 * pZ)
        post0 = _normalize((1.0 - q1) * pZ)
        mi[i] = p_r1 * _kl(post1, pZ) + p_r0 * _kl(post0, pZ)
    return mi

pool = np.random.default_rng(SEED + 3).uniform(-2.6, 2.6, size=(400, 2))
mi = mutual_information(x_init, pool, n_clusters=min(6, N_INIT), seed=SEED)
short_x = pool[np.argsort(-mi)[:24]]

def rbf_posterior(x_train, y_train, x_query, lengthscale=0.55, noise=0.12):
    def kern(a, b):
        d2 = np.sum((a[:, None, :] - b[None, :, :]) ** 2, axis=-1)
        return np.exp(-0.5 * d2 / lengthscale ** 2)
    k_tt = kern(x_train, x_train) + (noise ** 2) * np.eye(len(x_train))
    k_qt = kern(x_query, x_train)
    k_qq = kern(x_query, x_query)
    solved = np.linalg.solve(k_tt, k_qt.T)
    mean = k_qt @ np.linalg.solve(k_tt, y_train)
    var = np.clip(np.diag(k_qq) - np.sum(k_qt * solved.T, axis=1), 1e-8, None)
    return mean, var

gp_mean, gp_var = rbf_posterior(x_init, y_init, grid)
fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), constrained_layout=True)
im0 = heat(axes[0], grid, gp_mean, "Posterior mean from the real labels")
im1 = heat(axes[1], grid, np.sqrt(gp_var), "Posterior std")
for ax in axes:
    ax.scatter(x_init[:, 0], x_init[:, 1], c="white", edgecolors="k", s=22, zorder=3)
axes[1].scatter(short_x[:, 0], short_x[:, 1], marker="*", c="#FFD700", edgecolors="k", s=70, zorder=4, label="MI shortlist")
axes[1].legend(loc="lower right", fontsize=8)
fig.colorbar(im0, ax=axes[0], fraction=0.046)
fig.colorbar(im1, ax=axes[1], fraction=0.046)
plt.show()


## 6. Step 4 — control variates pick the batch

On the shortlist only, draw a small ball of designs. Pair posterior draws of the real score with proxy values, estimate

\(\mu(x) = \mathrm{mean}(f - \beta g) + \beta\,\mathrm{mean}(g_{\mathrm{unpaired}})\), \(\beta = \mathrm{cov}(g, f) / \mathrm{var}(g)\).

The next batch is the shortlist points with the highest corrected score. Gold stars are that batch. The other shortlist points stay drawn, colored by the corrected score.


In [ ]:
def local_ball(x0, radius, n, rng):
    direction = rng.normal(size=(n, x0.shape[0]))
    direction /= np.maximum(np.linalg.norm(direction, axis=1, keepdims=True), 1e-8)
    radius_draw = radius * np.sqrt(rng.random((n, 1)))
    return np.clip(x0 + radius_draw * direction, -3.0, 3.0)

def cv_scores(x_query, x_train, y_train, n_pair=24, k_unpaired=32, n_draws=8, radius=0.25, seed=0):
    rng = np.random.default_rng(seed)
    out = np.full(len(x_query), np.nan)
    for i, x0 in enumerate(x_query):
        paired = local_ball(x0, radius, n_pair, rng)
        unpaired = local_ball(x0, radius, k_unpaired, rng)
        g_paired = sim_mean(paired)
        g_unpaired = sim_mean(unpaired)
        mean, var = rbf_posterior(x_train, y_train, paired)
        vals = []
        for _ in range(n_draws):
            f = mean + np.sqrt(var) * rng.standard_normal(len(paired))
            if np.var(g_paired) < 1e-8:
                continue
            beta = np.cov(g_paired, f)[0, 1] / np.var(g_paired)
            mu = np.mean(f - beta * g_paired) + beta * np.mean(g_unpaired)
            vals.append(mu)
        if vals:
            out[i] = float(np.mean(vals))
    return out

cv = cv_scores(short_x, x_init, y_init, seed=SEED)
cv_plot = np.where(np.isfinite(cv), cv, np.nan)
chosen = short_x[np.argsort(-np.nan_to_num(cv, nan=-np.inf))[:5]]

fig, ax = plt.subplots(figsize=(5.4, 4.8), constrained_layout=True)
heat(ax, grid, real_mean(grid), "Control variate re-ranks the shortlist")
sc = ax.scatter(short_x[:, 0], short_x[:, 1], c=cv_plot, cmap="viridis", edgecolors="k", s=42, zorder=3)
ax.scatter(chosen[:, 0], chosen[:, 1], marker="*", c="#FFD700", edgecolors="k", s=160, zorder=4, label="next batch")
ax.scatter(x_init[:, 0], x_init[:, 1], c="white", edgecolors="k", s=22, zorder=2)
ax.legend(loc="lower right", fontsize=8)
fig.colorbar(sc, ax=ax, fraction=0.046, label="CV mean")
plt.show()
print("next batch (x1, x2):")
print(np.round(chosen, 3))


## 7. What to look for

- The proxy agrees with the real field on the right-hand diamond and understates the left-hand one.
- The first real labels usually miss both diamonds, so the posterior is still vague there.
- Mutual information spends the shortlist on designs the labels do not already explain.
- The control variate keeps a candidate only when the proxy correction still says the target score is high. That is the same two-step rule used on KITTI, with a 2-D design instead of a detection frame.
